In [3]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.metrics.pairwise import cosine_similarity
from scipy.stats import rankdata
import numpy as np
from collections import Counter, defaultdict
import os
from surprise import accuracy, Dataset, SVD, Reader
from surprise.model_selection import train_test_split
from surprise.model_selection import cross_validate

In [4]:
class CBFEngine:
    def __init__(self, data_path='data/cbf_df.csv', ratings_count_path='data/movies_ratings_count.csv'):
        self.df_cbf = pd.read_csv(data_path)

        lista_compilado = self.df_cbf['geral'].tolist()

        cvec = CountVectorizer(stop_words='english', ngram_range=(1, 2), min_df=1, max_df=.85)
        sf = cvec.fit_transform(lista_compilado)

        transformer = TfidfTransformer()
        tfidf_matrix = transformer.fit_transform(sf)

        self.cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

        df_ratings_count = pd.read_csv(ratings_count_path)
        list_ratings_count = np.array(df_ratings_count.loc[:, 'count_ratings'].to_list())
        log_ratings = np.log1p(list_ratings_count)
        self.pop_score = (log_ratings - log_ratings.min()) / (log_ratings.max() - log_ratings.min())

    def recomendar_filmes_perfil(self, user_id, n_recomendacoes=50, threshold=4.0,df_traing_path='data_spliting_hybrid/training.csv'):
        df_traing = pd.read_csv(df_traing_path)

        df_user = df_traing.loc[df_traing['userId'] == user_id]
        df_user = df_user.loc[df_user['rating'] >= threshold]
        list_id_movies = df_user['movieId'].to_list()

        indices_usuario = self.df_cbf[self.df_cbf['movie_id'].isin(list_id_movies)].index.tolist()

        if not indices_usuario:
            return pd.DataFrame()

        n_filmes = self.cosine_sim.shape[0]
        score_acumulado = np.zeros(n_filmes)

        for idx in indices_usuario:
            sim_scores_filme = self.cosine_sim[idx]
            score_acumulado = np.maximum(score_acumulado, sim_scores_filme)

        sim_scores_total = score_acumulado

        sim_norm = rankdata(sim_scores_total) / len(sim_scores_total)
        pop_norm = rankdata(self.pop_score) / len(self.pop_score)

        peso_conteudo = 0.9
        peso_pop = 0.1
        score_hibrido = (sim_norm * peso_conteudo) + (pop_norm * peso_pop)

        sim_scores = list(enumerate(score_hibrido))
        sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
        sim_scores_filtrados = [s for s in sim_scores if s[0] not in indices_usuario]

        top_sim_scores = sim_scores_filtrados[:n_recomendacoes]
        filme_indices = [i[0] for i in top_sim_scores]

        recomendacoes = self.df_cbf.iloc[filme_indices][['movie_id', 'title']].copy()
        recomendacoes['score_recomendacao'] = [i[1] for i in top_sim_scores]

        return recomendacoes

    def verifica_relevantes(self, user_id, recomendacoes, threshold=4.0,df_testing_path='data_spliting_hybrid/testing.csv'):
        df_testing = pd.read_csv(df_testing_path)
        recomendacoes = recomendacoes.loc[:, 'movie_id'].tolist()
        filmes_user_testing = df_testing.loc[
            (df_testing['userId'] == user_id) & (df_testing['rating'] >= threshold)
        ]
        total = pd.Series(recomendacoes).isin(filmes_user_testing['movieId']).sum()
        print(f'O total de filmes para o {user_id} relevantes é de: {total}')
        return total

    def validar_modelo_completo(self, df_test, k=10, threshold=4.0,df_train_path='data_spliting_hybrid/training.csv'):
        df_test_filtrado = df_test[df_test['rating'] >= threshold]
        usuarios_no_teste = df_test_filtrado['userId'].unique()
        df_train = pd.read_csv(df_train_path)

        total_precision = 0
        total_recall = 0
        contagem_usuarios = 0
        resultados_por_usuario = []

        for user_id in usuarios_no_teste:
            filmes_reais = set(df_test_filtrado[df_test_filtrado['userId'] == user_id]['movieId'])

            rec_df = self.recomendar_filmes_perfil(user_id, n_recomendacoes=k, threshold=threshold)

            if rec_df.empty:
                continue

            filmes_recomendados = set(rec_df['movie_id'])

            hits = len(filmes_recomendados.intersection(filmes_reais))

            precision_user = hits / k
            recall_user = hits / len(filmes_reais)

            total_precision += precision_user
            total_recall += recall_user
            contagem_usuarios += 1

            filmes_assistidos = df_train[
                (df_train['userId'] == user_id) & (df_train['rating'] >= threshold)
            ]['movieId'].tolist()

            resultados_por_usuario.append({
                'user_id': user_id,
                'precision': precision_user,
                'recall': recall_user,
                'hits': hits,
                'filmes_recomendados': rec_df.copy(),
                'filmes_assistidos': filmes_assistidos,
                'filmes_teste': list(filmes_reais),
            })

        metricas_finais = {
            'Mean Precision@K': total_precision / contagem_usuarios,
            'Mean Recall@K': total_recall / contagem_usuarios,
            'Usuarios Avaliados': contagem_usuarios
        }

        return metricas_finais, resultados_por_usuario

    def exibir_top_usuarios_por_precisao(self, resultados_por_usuario, top_n=10):
        top_usuarios = sorted(resultados_por_usuario, key=lambda x: x['precision'], reverse=True)[:top_n]

        for rank, user_data in enumerate(top_usuarios, start=1):
            user_id               = user_data['user_id']
            precision             = user_data['precision']
            recall                = user_data['recall']
            hits                  = user_data['hits']
            rec_df                = user_data['filmes_recomendados']
            filmes_assistidos_ids = user_data['filmes_assistidos']
            filmes_teste_ids      = set(user_data['filmes_teste'])

            print(f"\n{'='*75}")
            print(f"  TOP {rank:>2} | Usuário ID: {user_id} | Precision@10: {precision:.1%} | Recall@10: {recall:.1%} | Hits: {hits}")
            print(f"{'='*75}")

            assistidos_df = self.df_cbf[self.df_cbf['movie_id'].isin(filmes_assistidos_ids)][['movie_id', 'title']]
            print(f"\n  [FILMES ASSISTIDOS - TREINO]  ({len(assistidos_df)} filmes)")
            print(f"  {'-'*60}")
            for _, row in assistidos_df.iterrows():
                print(f"    {row['movie_id']:>7}  {row['title']}")

            teste_df = self.df_cbf[self.df_cbf['movie_id'].isin(filmes_teste_ids)][['movie_id', 'title']]
            print(f"\n  [FILMES DO TESTE - GABARITO]  ({len(teste_df)} filmes)")
            print(f"  {'-'*60}")
            for _, row in teste_df.iterrows():
                print(f"    {row['movie_id']:>7}  {row['title']}")

            indices_assistidos = self.df_cbf[self.df_cbf['movie_id'].isin(filmes_assistidos_ids)].index.tolist()
            indices_rec        = self.df_cbf[self.df_cbf['movie_id'].isin(rec_df['movie_id'])].index.tolist()

            score_por_idx = dict(zip(indices_rec, rec_df['score_recomendacao'].tolist()))

            print(f"\n  [FILMES RECOMENDADOS + VETORES DE SIMILARIDADE]")
            print(f"  (similaridade cosseno em relação aos {len(indices_assistidos)} filmes do perfil)")
            print(f"  {'-'*75}")
            print(f"  {'#':>2}  {'Título':<50}  {'Score':>7}  {'Sim Média':>9}  {'Sim Máx':>8}  {'Hit?':>5}")
            print(f"  {'-'*75}")

            for pos, (idx_rec, (_, row)) in enumerate(zip(indices_rec, rec_df.iterrows()), start=1):
                if indices_assistidos:
                    sim_vec = self.cosine_sim[idx_rec][indices_assistidos]
                    sim_med = float(sim_vec.mean())
                    sim_max = float(sim_vec.max())
                else:
                    sim_med = sim_max = 0.0

                hit_marker = ' HIT' if row['movie_id'] in filmes_teste_ids else '    '
                titulo = str(row['title'])[:50]
                print(f"  {pos:>2}. {titulo:<50}  {row['score_recomendacao']:>7.4f}  {sim_med:>9.4f}  {sim_max:>8.4f}  {hit_marker}")

        print(f"\n{'='*75}")

    def gerar_estudo_cobertura(self, df_test, ks=[100, 200, 300], threshold=4.0, output_dir='estudo_hibrido', df_train_path='data_spliting_hybrid/training.csv'):
        os.makedirs(output_dir, exist_ok=True)

        df_test_filtrado = df_test[df_test['rating'] >= threshold]

        for k in ks:
            resultados = []

            for user_id in df_test_filtrado['userId'].unique():
                filmes_reais = set(df_test_filtrado[df_test_filtrado['userId'] == user_id]['movieId'])

                rec_df = self.recomendar_filmes_perfil(user_id, n_recomendacoes=k, threshold=threshold,
                                                       df_traing_path=df_train_path)

                if rec_df.empty:
                    continue

                filmes_recomendados = set(rec_df['movie_id'])

                apareceram     = list(filmes_recomendados.intersection(filmes_reais))
                nao_apareceram = list(filmes_reais - filmes_recomendados)
                porcentagem    = round(len(apareceram) / len(filmes_reais) * 100, 2)

                resultados.append({
                    'userId': user_id,
                    'porcentagem_coberta': porcentagem,
                    'filmes_apareceram': apareceram,
                    'filmes_nao_apareceram': nao_apareceram,
                })

            df_resultado = pd.DataFrame(resultados)
            caminho = os.path.join(output_dir, f'cobertura_cbf_k{k}.csv')
            df_resultado.to_csv(caminho, index=False)
            print(f"[K={k}] Salvo em '{caminho}' — {len(df_resultado)} usuários | Cobertura média: {df_resultado['porcentagem_coberta'].mean():.2f}%")

In [5]:
cbf = CBFEngine()

In [6]:
df_teste = pd.read_csv('data_spliting_hybrid/testing.csv')
metricas, resultados_usuarios = cbf.validar_modelo_completo(df_teste)
print(metricas)

{'Mean Precision@K': 0.14809917355371888, 'Mean Recall@K': 0.0741103933708813, 'Usuarios Avaliados': 605}


In [7]:
class CFEngine:
    def __init__(self):
        self.reader = Reader(rating_scale=([0.5, 5]))

        self.df_ratings_total = pd.read_csv('ml-latest-small/ratings.csv').drop(columns='timestamp')
        self.df_ratings_traing = pd.read_csv('data_spliting_hybrid/training.csv').drop(columns='timestamp')
        self.df_ratings_testing = pd.read_csv('data_spliting_hybrid/testing.csv').drop(columns='timestamp')
        self.df_movies = pd.read_csv('ml-latest-small/movies.csv')

        self.data_training = Dataset.load_from_df(self.df_ratings_traing, reader=self.reader)
        self.data_testing = Dataset.load_from_df(self.df_ratings_testing, reader=self.reader)
        '''
        O Surprise espera dois formatos diferentes:

        Trainset → um objeto especial interno do Surprise, usado pelo .fit()
        Testset → uma lista de tuplas (uid, iid, nota_real), usado pelo .test()

        '''

        # Treino: build_full_trainset() — constrói um trainset com TODOS os dados do DataFrame
        self.trainset = self.data_training.build_full_trainset()

        # Teste: build_full_trainset().build_testset() — converte para lista de tuplas (uid, iid, nota_real)
        self.testset = self.data_testing.build_full_trainset().build_testset()
                

        self.algo = SVD(    
            n_factors=100,    # dimensões do espaço latente
            n_epochs=20,      # épocas de SGD
            lr_all=0.005,     # learning rate
            reg_all=0.02,     # regularização L2
            random_state=42,
            verbose=True
        )

        self.algo.fit(self.trainset)

        #### Algoítimo aprendido:
        print(f'vetor latente usuário: {self.algo.pu.shape}')
        print(f'vetor latente filme: {self.algo.qi.shape}')
        print(f'vetor de vies de cada usuário: {self.algo.bu.shape}')
        print(f'vetor de vies de cada filme: {self.algo.bi.shape}')

        '''
        Testar o modelo, comparando as predições feitas a partir do treinamento com os dados presentes no teste
        exemplo predição do user 1 no filme 133:
        Prediction(uid=1, iid=1030, r_ui=3.0, est=3.8121344471576815, details={'was_impossible': False}

        '''
        predicoes = self.algo.test(self.testset)

        self.filmes_no_treino = set(self.df_ratings_traing['movieId'].unique())
        self.usuarios_no_treino = set(self.df_ratings_traing['userId'].unique())
        

    def recomenda_usuario(self,usuario_id, top_n=10):
        filmes_assistidos = self.df_ratings_traing[self.df_ratings_traing['userId'] == usuario_id]['movieId'].tolist()

        filmes_nao_assistidos = sorted([
            movie_id for movie_id in self.filmes_no_treino
            if movie_id not in filmes_assistidos
        ])

        predicoes = []
        for id in filmes_nao_assistidos:
            pred = self.algo.predict(uid=usuario_id, iid=id)
            predicoes.append((id, pred.est))

        arr_sorted = sorted(predicoes, key=lambda x: x[1], reverse=True)

        recomendados = arr_sorted[:top_n]
        ids_recomendados = [movie_id for movie_id, nota in recomendados]

        df_resultado = self.df_movies.loc[self.df_movies['movieId'].isin(ids_recomendados), ['movieId', 'titulo_movie_lens']]

        return df_resultado
    
    def precisao_recall_usuario(self,usuario_id, top_n=10):
        ids_recomendados = self.recomenda_usuario(usuario_id, top_n)['movieId'].tolist()

        filmes_no_teste_usuario = self.df_ratings_testing[self.df_ratings_testing['userId'] == usuario_id]['movieId'].tolist()

        relevantes_recomendados = [m for m in ids_recomendados if m in filmes_no_teste_usuario]

        precisao = len(relevantes_recomendados) / len(ids_recomendados) if ids_recomendados else 0
        recall = len(relevantes_recomendados) / len(filmes_no_teste_usuario) if filmes_no_teste_usuario else 0

        print(f"Usuário: {usuario_id}")
        print(f"Filmes recomendados: {len(ids_recomendados)}")
        print(f"Filmes do usuário no teste: {len(filmes_no_teste_usuario)}")
        print(f"Relevantes recomendados: {len(relevantes_recomendados)}")
        print(f"Precisão: {precisao:.4f}")
        print(f"Recall:   {recall:.4f}")

        return precisao, recall

    def precision_recall_at_k_v2(self, k=50, threshold=4.0):
        """
        Calcula Precision@K e Recall@K chamando recomenda_usuario para cada usuário.
        Um filme é considerado relevante se estiver no teste com rating >= threshold.
        """
        usuarios = self.df_ratings_testing['userId'].unique()

        precisions = {}
        recalls = {}

        for usuario_id in usuarios:
            ids_recomendados = self.recomenda_usuario(usuario_id, top_n=k)['movieId'].tolist()

            filmes_teste_usuario = self.df_ratings_testing[self.df_ratings_testing['userId'] == usuario_id]
            filmes_relevantes_teste = set(
                filmes_teste_usuario[filmes_teste_usuario['rating'] >= threshold]['movieId'].tolist()
            )

            n_rel_top_k = sum(1 for movie_id in ids_recomendados if movie_id in filmes_relevantes_teste)
            n_rel_total = len(filmes_relevantes_teste)

            precisions[usuario_id] = n_rel_top_k / k if k > 0 else 0
            recalls[usuario_id] = n_rel_top_k / n_rel_total if n_rel_total > 0 else 0

        mean_precision = sum(precisions.values()) / len(precisions) if precisions else 0
        mean_recall = sum(recalls.values()) / len(recalls) if recalls else 0

        return mean_precision, mean_recall, precisions, recalls




                

In [8]:
cf_engine= CFEngine()
K=10
mean_p, mean_r, precisions, recalls = cf_engine.precision_recall_at_k_v2(k=K, threshold=4.0)


print(f'Mean Precision@{K}: {mean_p:.4f}')
print(f'Mean Recall@{K}:    {mean_r:.4f}')
print(f'Usuários avaliados: {len(precisions)}')


Processing epoch 0
Processing epoch 1
Processing epoch 2
Processing epoch 3
Processing epoch 4
Processing epoch 5
Processing epoch 6
Processing epoch 7
Processing epoch 8
Processing epoch 9
Processing epoch 10
Processing epoch 11
Processing epoch 12
Processing epoch 13
Processing epoch 14
Processing epoch 15
Processing epoch 16
Processing epoch 17
Processing epoch 18
Processing epoch 19
vetor latente usuário: (610, 100)
vetor latente filme: (8539, 100)
vetor de vies de cada usuário: (610,)
vetor de vies de cada filme: (8539,)
Mean Precision@10: 0.0810
Mean Recall@10:    0.0419
Usuários avaliados: 610


## Hybrid Engine — Cascade + Supplement

**Arquitetura:**
- **Cascade**: CBF gera top-K candidatos → CF re-rankeia via score unificado
- **Supplement**: filmes fora do top-K do CBF recebem só o score CF com desconto γ

**Fórmulas:**
```
# Filmes dentro do top-K do CBF (cascade):
score(i) = α · cbf_norm(i) + (1-α) · cf_norm_global(i)

# Filmes fora do top-K do CBF (supplement):
score(i) = γ · cf_norm_global(i)
```

**Normalização:**
- `cbf_norm(i)` = rank do filme entre os K candidatos CBF / K  (local)
- `cf_norm_global(i)` = rank do rating predito entre TODOS os não-assistidos / N  (global)

**Parâmetros:**
- `K` — tamanho do pool CBF (cobertura vs. custo)
- `α` — peso CBF vs. CF dentro do cascade
- `γ` — desconto do supplement (controla quantos itens CF puro entram no top-N)

In [9]:
class HybridEngine:
    def __init__(self, cbf_engine, cf_engine):
        self.cbf = cbf_engine
        self.cf = cf_engine
        self._title_map = dict(zip(self.cbf.df_cbf['movie_id'], self.cbf.df_cbf['title']))

    def recomendar(self, user_id, n_recomendacoes=10, K=200, alpha=0.6, gamma=0.75, threshold=4.0, df_train_path='data_spliting_hybrid/training.csv'):

        df_train = pd.read_csv(df_train_path)

        # --- CBF: top-K candidatos com scores ---
        cbf_rec = self.cbf.recomendar_filmes_perfil(
            user_id, n_recomendacoes=K, threshold=threshold, df_traing_path=df_train_path
        )
        cbf_movie_ids = set(cbf_rec['movie_id'].tolist()) if not cbf_rec.empty else set()

        # --- CF: ratings preditos para TODOS os filmes não assistidos ---
        filmes_assistidos = set(df_train[df_train['userId'] == user_id]['movieId'].tolist())
        filmes_nao_assistidos = [m for m in self.cf.filmes_no_treino if m not in filmes_assistidos]

        cf_predictions = {
            m: self.cf.algo.predict(uid=user_id, iid=m).est
            for m in filmes_nao_assistidos
        }

        # --- Normalização global do CF: rank / N_total ---
        all_ids = list(cf_predictions.keys())
        cf_ranks = rankdata([cf_predictions[m] for m in all_ids])
        N_total = len(all_ids)
        cf_norm_global = {m: cf_ranks[i] / N_total for i, m in enumerate(all_ids)}

        # --- Normalização local do CBF: rank / K ---
        cbf_norm_local = {}
        if not cbf_rec.empty:
            cbf_ranks = rankdata(cbf_rec['score_recomendacao'].tolist())
            for i, movie_id in enumerate(cbf_rec['movie_id']):
                cbf_norm_local[movie_id] = cbf_ranks[i] / K

        # --- Score final por origem ---
        scores_finais = {}
        for movie_id in filmes_nao_assistidos:
            cf_n = cf_norm_global.get(movie_id, 0)
            if movie_id in cbf_movie_ids:
                # cascade: sinal de conteúdo + colaborativo
                scores_finais[movie_id] = alpha * cbf_norm_local[movie_id] + (1 - alpha) * cf_n
            else:
                # supplement: só CF, com desconto γ
                scores_finais[movie_id] = gamma * cf_n

        # --- Top-N final ---
        top_n = sorted(scores_finais.items(), key=lambda x: x[1], reverse=True)[:n_recomendacoes]

        rows = []
        for movie_id, score in top_n:
            title = self._title_map.get(movie_id, f'[id={movie_id}]')
            origem = 'cascade' if movie_id in cbf_movie_ids else 'supplement'
            rows.append({'movie_id': movie_id, 'title': title, 'score_hibrido': score, 'origem': origem})

        return pd.DataFrame(rows)

    def validar_modelo_completo(self, df_test, n=10, K=200, alpha=0.6, gamma=0.75,
                                threshold=4.0, df_train_path='data_spliting_hybrid/training.csv'):

        df_test_filtrado = df_test[df_test['rating'] >= threshold]
        usuarios = df_test_filtrado['userId'].unique()

        total_precision = 0
        total_recall = 0
        contagem = 0
        resultados = []

        for user_id in usuarios:
            filmes_reais = set(df_test_filtrado[df_test_filtrado['userId'] == user_id]['movieId'])

            rec_df = self.recomendar(
                user_id, n_recomendacoes=n, K=K, alpha=alpha, gamma=gamma,
                threshold=threshold, df_train_path=df_train_path
            )
            if rec_df.empty:
                continue

            hits = len(set(rec_df['movie_id']) & filmes_reais)
            precision = hits / n
            recall = hits / len(filmes_reais)

            total_precision += precision
            total_recall += recall
            contagem += 1

            resultados.append({
                'user_id': user_id,
                'precision': precision,
                'recall': recall,
                'hits': hits,
                'n_cascade': (rec_df['origem'] == 'cascade').sum(),
                'n_supplement': (rec_df['origem'] == 'supplement').sum(),
                'filmes_recomendados': rec_df.copy(),
                'filmes_reais': list(filmes_reais),
            })

        metricas = {
            'Mean Precision@K': round(total_precision / contagem, 4),
            'Mean Recall@K': round(total_recall / contagem, 4),
            'Usuarios Avaliados': contagem,
            'Params': {'K': K, 'alpha': alpha, 'gamma': gamma}
        }
        return metricas, resultados

In [10]:
hybrid = HybridEngine(cbf_engine=cbf, cf_engine=cf_engine)

# Exemplo: recomendação para um usuário específico
rec = hybrid.recomendar(user_id=1, n_recomendacoes=10, K=200, alpha=0.6, gamma=0.75)
print(rec.to_string(index=False))

 movie_id                                                            title  score_hibrido  origem
     1210                Star Wars: Episode VI - Return of the Jedi (1983)       0.996944 cascade
     4993        Lord of the Rings: The Fellowship of the Ring, The (2001)       0.995520 cascade
     7153            Lord of the Rings: The Return of the King, The (2003)       0.979183 cascade
     5952                    Lord of the Rings: The Two Towers, The (2002)       0.977361 cascade
      589                                Terminator 2: Judgment Day (1991)       0.943844 cascade
     1200                                                    Aliens (1986)       0.924594 cascade
      296                                              Pulp Fiction (1994)       0.906475 cascade
     2788 Monty Python's And Now for Something Completely Different (1971)       0.901112 cascade
    33794                                             Batman Begins (2005)       0.892368 cascade
     6807           

In [11]:
df_teste = pd.read_csv('data_spliting_hybrid/testing.csv')

metricas_hibrido, resultados_hibrido = hybrid.validar_modelo_completo(
    df_teste, n=10, K=200, alpha=0.6, gamma=0.75
)

print("=" * 50)
print("COMPARATIVO DE MODELOS — Precision@10 / Recall@10")
print("=" * 50)
print(f"  CBF puro   →  P: {0.1481:.4f}  |  R: {0.0741:.4f}")
print(f"  CF puro    →  P: {0.0810:.4f}  |  R: {0.0419:.4f}")
print(f"  Híbrido    →  P: {metricas_hibrido['Mean Precision@K']:.4f}  |  R: {metricas_hibrido['Mean Recall@K']:.4f}")
print(f"  Params: K={metricas_hibrido['Params']['K']}, α={metricas_hibrido['Params']['alpha']}, γ={metricas_hibrido['Params']['gamma']}")
print("=" * 50)

# Distribuição cascade vs supplement nos top-10
n_cascade_medio = np.mean([r['n_cascade'] for r in resultados_hibrido])
n_supplement_medio = np.mean([r['n_supplement'] for r in resultados_hibrido])
print(f"\nDistribuição média no top-10:")
print(f"  Cascade (CBF→CF):  {n_cascade_medio:.2f} slots")
print(f"  Supplement (CF):   {n_supplement_medio:.2f} slots")

COMPARATIVO DE MODELOS — Precision@10 / Recall@10
  CBF puro   →  P: 0.1481  |  R: 0.0741
  CF puro    →  P: 0.0810  |  R: 0.0419
  Híbrido    →  P: 0.1644  |  R: 0.0803
  Params: K=200, α=0.6, γ=0.75

Distribuição média no top-10:
  Cascade (CBF→CF):  9.93 slots
  Supplement (CF):   0.07 slots
